In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
import pandas as pd

books = pd.read_csv("books_cleaned.csv")
books

In [ ]:
books["tagged_description"]

In [ ]:
books["tagged_description"].to_csv("tagged_description.txt",sep="\n", index=False, header=False)

In [ ]:
raw_doc = TextLoader("tagged_description.txt", encoding="utf-8").load()
text_splitter = CharacterTextSplitter(chunk_size=0, chunk_overlap=0, separator="\n")
docs = text_splitter.split_documents(raw_doc)

In [ ]:
docs[0]

In [ ]:
db_books = Chroma.from_documents(docs, embedding=OpenAIEmbeddings())

In [ ]:
query = "A book to teach children about nature"
doc = db_books.similarity_search(query, k=10)
doc

In [ ]:
books[books["isbn13"] == int(doc[0].page_content.split()[0].strip())]

In [ ]:
def retrieve_semantic_recommendations(query: str, top_k: int=10) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k=10)
    books_list = []
    for i in range(0, len(recs)):
        books_list += [int(recs[i].page_content.strip('"').split()[0])]
    return books[books["isbn13"].isin(books_list)].head(top_k)

In [ ]:
retrieve_semantic_recommendations("A book to teach children about nature")